In [0]:

from databricks.feature_store import FeatureStoreClient
import pyspark.sql.functions as F

fe = FeatureStoreClient()

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
raw_table_name = f"{current_catalog}.default.wine_quality_raw"
feature_table_name = f"{current_catalog}.default.wine_features"

df_raw = spark.table(raw_table_name)

df_prepared = df_raw.withColumn("id_sample", F.monotonically_increasing_id())

df_features = df_prepared \
    .withColumn("alcohol_density_ratio", F.col("alcohol") / F.col("density")) \
    .withColumn("total_acidity", F.col("fixed_acidity") + F.col("volatile_acidity")) \
    .withColumn("is_high_quality", F.when(F.col("quality") >= 6, 1).otherwise(0))

spark.sql(f"DROP TABLE IF EXISTS {feature_table_name}")

fe.create_table(
    name=feature_table_name,
    primary_keys=["id_sample"],
    df=df_features,
    description="Tabla de características procesadas para entrenamiento de modelo"
)

print(f"Tabla '{feature_table_name}' registrada exitosamente en Feature Store.")